In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import plotly.graph_objects as go
from fastapi import APIRouter, Query
import plotly.express as px

# Database connection details
load_dotenv()

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# Create the connection engine
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

In [17]:
def scam_loss(state: str = "ALL"):
    query = """
    SELECT "Scam_Type", SUM("Amount_Lost") AS "Total_Lost"
    FROM data_insight
    WHERE (%(state)s = 'ALL' OR "Address_State" = %(state)s)
        AND "Complainant_Age" = '65 and over'
    GROUP BY "Scam_Type"
    ORDER BY "Total_Lost"
    LIMIT 3;
    """
    df = pd.read_sql(query, engine, params={"state": state})
    df["Total_Lost"] = df["Total_Lost"].round(2)
    return df.to_dict(orient="records")
scam_loss()


[{'Scam_Type': 'Job Scams', 'Total_Lost': 869364.63},
 {'Scam_Type': 'E-commerce Scams', 'Total_Lost': 14559116.62},
 {'Scam_Type': 'Romance Scams', 'Total_Lost': 23615327.32}]

In [18]:
def contact_mode(state: str = "ALL"):
    query = """
    SELECT "Contact_Mode", SUM("Number_of_reports") AS "Number_of_reports"
    FROM data_insight
    WHERE (%(state)s = 'ALL' OR "Address_State" = %(state)s)
        AND "Complainant_Age" = '65 and over'
    GROUP BY "Contact_Mode"
    ORDER BY "Number_of_reports" DESC
    LIMIT 3;
    """
    df = pd.read_sql(query, engine, params={"state": state})
    df["Number_of_reports"] = df["Number_of_reports"].astype(int)
    return df.to_dict(orient="records")
contact_mode()

[{'Contact_Mode': 'Email', 'Number_of_reports': 51538},
 {'Contact_Mode': 'SMS', 'Number_of_reports': 25187},
 {'Contact_Mode': 'Phone', 'Number_of_reports': 23785}]

In [19]:
def fig_to_html(fig):
    fig.update_layout(hovermode=False)
    for trace in fig.data:
        trace.update(hoverinfo="skip")
    return fig.to_html(include_plotlyjs="cdn", full_html=False,
                       config={"displayModeBar": False})

In [ ]:
def line_chart(state: str = "ALL"):
    query = """
    SELECT "Year", SUM("Number_of_reports") / COUNT(DISTINCT "Month") AS avg_reports
    FROM data_insight
    WHERE (%(state)s = 'ALL' OR "Address_State" = %(state)s)
        AND "Complainant_Age" = '65 and over'
    GROUP BY "Year"
    ORDER BY "Year";
    """
    df = pd.read_sql(query, engine, params={"state": state})

    # Initial trace (first point)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=[df["Year"].iloc[0]],
        y=[df["avg_reports"].iloc[0]],
        mode="lines+markers",
        line=dict(width=4, color="#1E3A8A"),
        marker=dict(size=10, color="#764ba2")
    ))

    # Frames: gradually add one more year per frame
    frames = []
    for k in range(1, len(df) + 1):
        frames.append(go.Frame(
            data=[go.Scatter(
                x=df["Year"].iloc[:k],
                y=df["avg_reports"].iloc[:k],
                mode="lines+markers",
                line=dict(width=4, color="#667eea"),
                marker=dict(size=10, color="#764ba2")
            )],
            name=str(k)
        ))

    fig.frames = frames

    # Layout
    fig.update_layout(
        title=dict(
            text=f"Scams Reported per Month from 2021 - 2025 ({state})",
            x=0.5,                # center horizontally
            xanchor="center",
            font=dict(size=22)    # bigger title font
        ),
        xaxis=dict(
            title=dict(text="Year", font=dict(size=20)),
            range=[df["Year"].min()-0.5, df["Year"].max()+0.5]
        ),
        yaxis=dict(
            title=dict(text="Reports", font=dict(size=20)),
            range=[0, df["avg_reports"].max() * 1.1]
        ),
        showlegend=False,
        updatemenus=[{
            "type": "buttons",
            "showactive": False,
            "buttons": [{
                "label": "Play",
                "method": "animate",
                "args": [None, {
                    "frame": {"duration": 600, "redraw": True},
                    "transition": {"duration": 600, "easing": "cubic-in-out"},
                    "fromcurrent": True,
                    "mode": "immediate"
                }]
            }]
        }],
        hovermode=False,
        dragmode=False
    )
    return fig
    
line_chart()

'<div>                        <script type="text/javascript">window.PlotlyConfig = {MathJaxConfig: \'local\'};</script>\n        <script src="https://cdn.plot.ly/plotly-2.18.2.min.js"></script>                <div id="9982cf58-677a-4bdb-bebc-48a1b4cc8fb6" class="plotly-graph-div" style="height:100%; width:100%;"></div>            <script type="text/javascript">                                    window.PLOTLYENV=window.PLOTLYENV || {};                                    if (document.getElementById("9982cf58-677a-4bdb-bebc-48a1b4cc8fb6")) {                    Plotly.newPlot(                        "9982cf58-677a-4bdb-bebc-48a1b4cc8fb6",                        [{"line":{"color":"#1E3A8A","width":4},"marker":{"color":"#764ba2","size":10},"mode":"lines+markers","x":[2021],"y":[1065.5],"type":"scatter","hoverinfo":"skip"}],                        {"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.

In [23]:
def pie_chart(state: str = "ALL"):
    query = """
    SELECT "Complainant_Gender", COUNT(*) AS count
    FROM data_insight
    WHERE (%(state)s = 'ALL' OR "Address_State" = %(state)s)
        AND "Complainant_Age" = '65 and over'
    GROUP BY "Complainant_Gender";
    """
    df = pd.read_sql(query, engine, params={"state": state})

    # Build pie chart
    fig = go.Figure(data=[go.Pie(
        labels=df["Complainant_Gender"],
        values=df["count"],
        hole=0.4,  
        textinfo="label+percent",
        insidetextorientation="radial",
        marker=dict(colors=["#764ba2", "#1E3A8A"])
    )])

    # Layout
    fig.update_layout(
        title=dict(
            text=f"Scam Victims by Gender between 2021 and 2025 ({state})",
            x=0.5,
            xanchor="center",
            font=dict(size=22)
        ),
        legend=dict(
            title="Gender",
            font=dict(size=16)
        ),
        showlegend=False,
        hovermode=False,
        template=None
    )

    return fig

pie_chart()


In [13]:
def bar_chart(state: str = "ALL"):
    query = """
    SELECT "Scam_Type", "Complainant_Gender", SUM("Number_of_reports") AS reports
    FROM data_insight
    WHERE (%(state)s = 'ALL' OR "Address_State" = %(state)s)
        AND "Complainant_Age" = '65 and over'
    GROUP BY "Scam_Type", "Complainant_Gender";
    """
    df = pd.read_sql(query, engine, params={"state": state})

    # Rank scam types by total reports
    top3 = (
        df.groupby("Scam_Type")["reports"]
        .sum()
        .nlargest(3)
        .index
    )
    df_top3 = df[df["Scam_Type"].isin(top3)]

    # Pivot table (Male vs Female counts)
    pivot_df = df_top3.pivot_table(
        index="Scam_Type",
        columns="Complainant_Gender",
        values="reports",
        fill_value=0
    ).reset_index()

    # Add total column for sorting
    pivot_df["total"] = pivot_df.sum(axis=1, numeric_only=True)
    pivot_df = pivot_df.sort_values("total", ascending=False)

    scam_types = pivot_df["Scam_Type"].tolist()
    male_counts = pivot_df.get("Male", pd.Series([0]*len(pivot_df))).tolist()
    female_counts = pivot_df.get("Female", pd.Series([0]*len(pivot_df))).tolist()

    # Start from 0
    fig = go.Figure(data=[
        go.Bar(name="Male", x=scam_types, y=[0]*len(scam_types), marker_color="#1E3A8A"),
        go.Bar(name="Female", x=scam_types, y=[0]*len(scam_types), marker_color="#764ba2")
    ])

    # Build frames (bars grow step by step)
    frames = []
    steps = 150
    for step in range(1, steps+1):
        frame_male = [v * step/steps for v in male_counts]
        frame_female = [v * step/steps for v in female_counts]
        frames.append(go.Frame(
            data=[
                go.Bar(name="Male", x=scam_types, y=frame_male, marker_color="#1E3A8A"),
                go.Bar(name="Female", x=scam_types, y=frame_female, marker_color="#764ba2")
            ],
            name=str(step)
        ))
    fig.frames = frames

    # Layout
    fig.update_layout(
        title=dict(
            text=f"Top 3 Scam Types by Gender between 2021 and 2025 ({state})",
            x=0.5, xanchor="center", font=dict(size=22)
        ),
        xaxis=dict(
            title=dict(text="Scam Type", font=dict(size=18)),
            tickfont=dict(size=16)
        ),
        yaxis=dict(
            title=dict(text="Reports", font=dict(size=18)),
            tickfont=dict(size=14),
            range=[0, max(male_counts + female_counts) * 1.1]
        ),
        barmode="group",
        bargap=0.25,
        legend=dict(font=dict(size=16)),
        updatemenus=[{
            "type": "buttons",
            "showactive": False,
            "buttons": [{
                "label": "Play",
                "method": "animate",
                "args": [None, {
                    "frame": {"duration": 10, "redraw": True},
                    "transition": {"duration": 10, "easing": "cubic-in-out"},
                    "mode": "immediate"
                }]
            }]
        }],
        hovermode=False,
        dragmode=False
    )

    return fig

bar_chart()